# Day 56：AI 应用容器化部署工程化与运维巡检（Jupyter Notebook 实战手册）

> **所属阶段**：第 5 阶段 · 第五阶段：模型微调与部署 ｜ **第 8 周**：模型微调、本地推理部署与安全合规  
> **GitLab 工单**：`STARMIND-056` ｜ **开发分支**：`feat/day56-docker-nginx-ops`  
> **配套 HTML5 课件**：`courseware/day56/index.html` ｜ **独立版本快照**：`versions/day56/`

---

## 🎙️ 老陈开场旁白

早啊各位！模型跑通了、压测也达标了，今天解决'在我电脑上好好的，一上阿里云 ECS 就炸'这个千古难题——AI 应用部署工程化。记住在大厂做 AI 应用上线，有两个最隐蔽的坑：第一个是 Dockerfile 直接 `FROM python:3.11` 傻打一通，镜像飙到 3GB，而且以 root 用户跑容器被安全组直接打回；第二个是 Nginx 反向代理 FastAPI 的 SSE 流式输出时，忘了关 `proxy_buffering off;`，结果前端死活收不到打字机效果，干等 30 秒一次性全吐出来！今天咱手写一套部署资产生成与校验器（多阶段 Dockerfile + Docker Compose + SSE 优化版 Nginx 配置）外加一个生产级运维健康巡检脚本。

### 承上启下
- **昨日基线**：昨天我们打通了 Ollama/vLLM 统一 OpenAI 推理协议，并手写了 TTFT/TPS 异步压测器验证了并发吞吐。
- **明日预告**：明天（Day 57）是第五阶段周测 + 安全与合规专题：内容审核、敏感信息脱敏、国内大模型网信办备案常识，以及 Token 成本实时监控与熔断。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day56'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-056）

星辰智脑即将部署至杭州阿里云 ECS 生产环境，需将 FastAPI 后端、前端控制台、Redis 缓存与上游推理服务统一容器化编排，并通过 Nginx 暴露 HTTPS/SSE 入口，同时配备上线后自动化巡检探针。

**今日迭代目标：**
- 实现部署配置生成与安全审计器：自动生成多阶段瘦身 Dockerfile、Docker Compose 编排文件与支持 SSE 流式无缓冲的 Nginx 配置，并校验关键安全与超时配置。
- 实现生产运维健康巡检引擎：并发检测多服务探针状态、响应延迟、磁盘/内存水位并输出红黄绿巡检报告。

**验收标准：**
- 生成的 Dockerfile 包含多阶段 `AS builder`、非 root 用户切换 `USER appuser` 与 `HEALTHCHECK` 指令。
- 生成的 Nginx 配置包含 `proxy_buffering off`、`proxy_cache off`、`proxy_read_timeout 300s` 及 HTTP/1.1 长连接配置。
- 运维巡检器准确识别健康、降级、宕机三种状态并返回标准退出码。

---

## 📚 课堂核心技术笔记

### 1. 1. Nginx 代理大模型 SSE 接口的 5 行保命配置

在 Nginx 的 `location /api/` 块中务必加上：`proxy_http_version 1.1; proxy_set_header Connection ''; proxy_buffering off; proxy_cache off; proxy_read_timeout 300s;`。此外，FastAPI 响应头中最好加上 `X-Accel-Buffering: no`，双保险禁止代理层攒包。

> **⚠️ 老陈避坑**：坑点：本地 `uvicorn` 调试时流式输出丝滑无比，一挂上阿里云 SLB + Nginx 就卡住 20 秒一起蹦出来，99% 都是漏了 `proxy_buffering off`。

### 2. 2. Docker Compose 启动顺序与 `depends_on` 陷阱

单纯写 `depends_on: [redis]` 只保证 Redis 容器'启动了进程'，不保证 Redis 已经'能接受连接'。生产写法必须配合 `healthcheck` 和 `condition: service_healthy`，确保底层存储完全就绪后再拉起 FastAPI。

> **⚠️ 老陈避坑**：坑点：把 `.env` 密钥文件直接 `COPY . .` 打进镜像层，一旦镜像推送到仓库，所有 API Key 全部裸奔泄露！务必在 `.dockerignore` 中排除 `.env`、`.git`、`.venv`。

### 3. 3. 阿里云/腾讯云服务器上线安全基线

安全组仅开放 80/443（Nginx）和受限 IP 的 22 端口，严禁将 FastAPI 8000、vLLM 8001、Redis 6379 直接暴露在公网！所有内部组件全部绑定在 Docker 私有桥接网络 `starmind-net` 内互通。

> **⚠️ 老陈避坑**：坑点：把未设密码的 Redis 或 Ollama 11434 暴露在公网，不到 10 分钟就会被全网扫描器白嫖算力甚至植入挖矿脚本。


## 💻 实操步骤 1：步骤一：实现多阶段 Dockerfile / Compose / SSE Nginx 生成与安全审计器

- **目标文件**：`versions/day56/src/day56_deploy_generator.py`
- **老陈代码旁白**：先把上线要用的三件套——Dockerfile、docker-compose.yml 和 nginx.conf——用代码规范化生成，再配一个安全审计函数，漏了非 root 用户或 SSE 关缓冲直接报错。


In [ ]:
"""
Day 56 - 模块一：容器化部署资产生成器（Dockerfile / Compose / Nginx SSE）与安全审计器
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import List


@dataclass
class DeploySpec:
    """部署环境规格"""
    app_name: str = "starmind-api"
    app_port: int = 8000
    nginx_port: int = 80
    workers: int = 2
    pip_mirror: str = "https://pypi.tuna.tsinghua.edu.cn/simple"


def render_multistage_dockerfile(spec: DeploySpec) -> str:
    """生成多阶段瘦身、非 root、带健康检查的生产级 Dockerfile"""
    return f"""# Stage 1: 构建依赖虚拟环境
FROM python:3.11-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN python -m venv /opt/venv \\
    && /opt/venv/bin/pip install --no-cache-dir -i {spec.pip_mirror} -r requirements.txt

# Stage 2: 极简安全运行时
FROM python:3.11-slim AS runtime
ENV PATH="/opt/venv/bin:$PATH" PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
RUN useradd -m -u 10001 starmind
WORKDIR /app
COPY --from=builder /opt/venv /opt/venv
COPY --chown=starmind:starmind . /app
USER starmind
EXPOSE {spec.app_port}
HEALTHCHECK --interval=30s --timeout=5s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:{spec.app_port}/healthz', timeout=3)" || exit 1
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "{spec.app_port}", "--workers", "{spec.workers}"]
"""


def render_sse_nginx_conf(spec: DeploySpec) -> str:
    """生成针对大模型 SSE 流式输出优化的 Nginx 反向代理配置"""
    return f"""server {{
    listen {spec.nginx_port};
    server_name _;

    client_max_body_size 50M;

    location /api/ {{
        proxy_pass http://{spec.app_name}:{spec.app_port};
        proxy_http_version 1.1;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header Connection '';
        proxy_buffering off;
        proxy_cache off;
        chunked_transfer_encoding on;
        proxy_read_timeout 300s;
    }}

    location /healthz {{
        proxy_pass http://{spec.app_name}:{spec.app_port}/healthz;
    }}
}}
"""


def audit_deploy_assets(dockerfile: str, nginx_conf: str) -> List[str]:
    """审计 Dockerfile 与 Nginx 配置，返回缺失的安全/性能红线项列表"""
    issues: List[str] = []
    if "AS builder" not in dockerfile:
        issues.append("Dockerfile 未使用多阶段构建(AS builder)，镜像体积将偏大")
    if "USER " not in dockerfile or "USER root" in dockerfile:
        issues.append("Dockerfile 未切换非 root 运行用户，存在容器提权风险")
    if "HEALTHCHECK" not in dockerfile:
        issues.append("Dockerfile 缺少 HEALTHCHECK 探针配置")
    if "proxy_buffering off;" not in nginx_conf:
        issues.append("Nginx 未关闭 proxy_buffering，将导致大模型 SSE 流式输出卡顿")
    if "proxy_read_timeout" not in nginx_conf:
        issues.append("Nginx 未设置长读超时，长文本生成易触发 504 Gateway Timeout")
    return issues


if __name__ == "__main__":
    spec = DeploySpec()
    df = render_multistage_dockerfile(spec)
    ngx = render_sse_nginx_conf(spec)
    problems = audit_deploy_assets(df, ngx)
    print(f"部署配置安全审计完成，发现问题数: {len(problems)}")


### 🔍 步骤 1 代码深度解析

把多阶段构建、非 root 用户、HEALTHCHECK 和 Nginx SSE 防缓冲全部固化为代码模板与自动化审计规则，彻底避免人工拷贝配置漏项。

**预期运行输出参考：**
```text
部署配置安全审计完成，发现问题数: 0
```


## 💻 实操步骤 2：步骤二：实现生产级运维健康巡检与分级告警引擎

- **目标文件**：`versions/day56/src/day56_ops_inspector.py`
- **老陈代码旁白**：服务上线后不能当甩手掌柜。我们写一个轻量级运维巡检器，每次发布完或每 5 分钟跑一遍，把 API、推理引擎、缓存的探针延迟和磁盘/内存水位汇总成 GREEN/YELLOW/RED 巡检报告。


In [ ]:
"""
Day 56 - 模块二：生产环境运维健康巡检与分级告警脚本
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List


@dataclass
class ServiceProbe:
    """单个依赖服务的探针采集结果"""
    name: str
    is_critical: bool
    healthy: bool
    latency_ms: float
    detail: str = ""


@dataclass
class HostMetrics:
    """宿主机资源水位"""
    cpu_percent: float
    mem_percent: float
    disk_percent: float


@dataclass
class InspectionReport:
    """运维巡检总报告"""
    overall_status: str     # "GREEN" | "YELLOW" | "RED"
    exit_code: int          # 0=健康, 1=警告降级, 2=严重故障
    alerts: List[str] = field(default_factory=list)
    summary: Dict[str, object] = field(default_factory=dict)


def run_ops_inspection(
    probes: List[ServiceProbe],
    host: HostMetrics,
    max_latency_ms: float = 500.0,
) -> InspectionReport:
    """评估所有微服务探针与主机水位，输出红黄绿分级状态与退出码"""
    alerts: List[str] = []
    has_red = False
    has_yellow = False

    for p in probes:
        if not p.healthy:
            if p.is_critical:
                has_red = True
                alerts.append(f"[CRITICAL] 核心服务 {p.name} 不可达: {p.detail}")
            else:
                has_yellow = True
                alerts.append(f"[WARN] 非核心服务 {p.name} 异常，已进入降级模式: {p.detail}")
        elif p.latency_ms > max_latency_ms:
            has_yellow = True
            alerts.append(f"[WARN] 服务 {p.name} 探针延迟偏高 ({p.latency_ms:.1f}ms > {max_latency_ms}ms)")

    if host.disk_percent >= 90.0 or host.mem_percent >= 92.0:
        has_red = True
        alerts.append(f"[CRITICAL] 宿主机资源危急: MEM={host.mem_percent}%, DISK={host.disk_percent}%")
    elif host.disk_percent >= 80.0 or host.mem_percent >= 82.0 or host.cpu_percent >= 85.0:
        has_yellow = True
        alerts.append(
            f"[WARN] 宿主机负载偏高: CPU={host.cpu_percent}%, MEM={host.mem_percent}%, DISK={host.disk_percent}%"
        )

    if has_red:
        status, code = "RED", 2
    elif has_yellow:
        status, code = "YELLOW", 1
    else:
        status, code = "GREEN", 0

    return InspectionReport(
        overall_status=status,
        exit_code=code,
        alerts=alerts,
        summary={
            "total_services": len(probes),
            "healthy_services": sum(1 for p in probes if p.healthy),
            "host": {"cpu": host.cpu_percent, "mem": host.mem_percent, "disk": host.disk_percent},
        },
    )


if __name__ == "__main__":
    probes = [
        ServiceProbe("starmind-api", True, True, 18.5, "ok"),
        ServiceProbe("vllm-gateway", True, True, 45.2, "ok"),
        ServiceProbe("redis-cache", False, True, 3.1, "ok"),
    ]
    rep = run_ops_inspection(probes, HostMetrics(cpu_percent=35.0, mem_percent=58.0, disk_percent=42.0))
    print(f"巡检状态: {rep.overall_status} (exit_code={rep.exit_code}), 告警数={len(rep.alerts)}")


### 🔍 步骤 2 代码深度解析

严格区分 `is_critical` 核心服务（如 FastAPI/vLLM）与非核心服务（如 Redis 缓存可降级直查数据库），并按照 Linux 监控惯例（Nagios/Prometheus 风格）返回 0/1/2 退出码，方便直接接入 Crontab 与钉钉告警机器人。

**预期运行输出参考：**
```text
巡检状态: GREEN (exit_code=0), 告警数=0
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day56/tests/test_day56.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 56 单元测试：部署资产审计与运维健康巡检"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day56_deploy_generator import (
    DeploySpec,
    audit_deploy_assets,
    render_multistage_dockerfile,
    render_sse_nginx_conf,
)
from src.day56_ops_inspector import HostMetrics, ServiceProbe, run_ops_inspection


def test_generated_deploy_assets_pass_security_audit():
    spec = DeploySpec()
    issues = audit_deploy_assets(render_multistage_dockerfile(spec), render_sse_nginx_conf(spec))
    assert issues == []


def test_audit_catches_root_user_and_missing_sse_buffering_off():
    bad_df = "FROM python:3.11\nUSER root\nCMD ['python']"
    bad_ngx = "location /api/ { proxy_pass http://api:8000; }"
    issues = audit_deploy_assets(bad_df, bad_ngx)
    assert len(issues) >= 4


def test_ops_inspection_green_yellow_red():
    ok_host = HostMetrics(30.0, 50.0, 40.0)
    r_green = run_ops_inspection([ServiceProbe("api", True, True, 20.0)], ok_host)
    assert r_green.overall_status == "GREEN" and r_green.exit_code == 0

    r_yellow = run_ops_inspection(
        [ServiceProbe("api", True, True, 20.0), ServiceProbe("redis", False, False, 0.0, "timeout")],
        ok_host,
    )
    assert r_yellow.overall_status == "YELLOW" and r_yellow.exit_code == 1

    r_red = run_ops_inspection([ServiceProbe("api", True, False, 0.0, "502")], ok_host)
    assert r_red.overall_status == "RED" and r_red.exit_code == 2

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：生成带健康检查依赖的 docker-compose.yml

**题目要求：**
编写函数 `render_docker_compose(spec: DeploySpec) -> str`，编排 `starmind-api` 与 `nginx` 两个服务，要求 `nginx` 通过 `depends_on` 等待 `starmind-api` 健康后再启动。

**思路提示：** 使用 `condition: service_healthy` 并声明内部桥接网络 `starmind-net`。


In [ ]:
from src.day56_deploy_generator import DeploySpec


def render_docker_compose(spec: DeploySpec) -> str:
    return f"""services:
  {spec.app_name}:
    build: .
    restart: unless-stopped
    env_file: .env
    networks: [starmind-net]
  nginx:
    image: nginx:1.27-alpine
    ports: ["{spec.nginx_port}:80"]
    volumes: ["./nginx.conf:/etc/nginx/conf.d/default.conf:ro"]
    depends_on:
      {spec.app_name}:
        condition: service_healthy
    networks: [starmind-net]
networks:
  starmind-net:
    driver: bridge
"""


print(render_docker_compose(DeploySpec()))


**答案解析与评分标准：**

只暴露 Nginx 的 80/443 端口，FastAPI 仅在 `starmind-net` 内网可见，符合企业最小暴露面原则。


### 作业 2（企业实战）：企业实战：编写 `.dockerignore` 生成器防止敏感文件泄露

**题目要求：**
编写审计函数检测 `.dockerignore` 是否包含 `.env`、`.git`、`.venv`、`__pycache__` 和 `*.key` 五项必备忽略规则。

**思路提示：** 按行拆分 `.dockerignore` 内容转成集合，求差集返回缺失项。


In [ ]:
REQUIRED_IGNORES = {".env", ".git", ".venv", "__pycache__", "*.key"}


def check_dockerignore(content: str) -> list[str]:
    lines = {line.strip() for line in content.splitlines() if line.strip() and not line.startswith("#")}
    return sorted(REQUIRED_IGNORES - lines)


print("缺失项:", check_dockerignore(".env\n.git\n.venv\n__pycache__\n*.key\n"))


**答案解析与评分标准：**

将 `.dockerignore` 检查加入 CI pre-commit 钩子，杜绝开发者误把私钥和本地虚拟环境打进镜像。


### 作业 3（架构进阶）：架构进阶：实现零停机蓝绿发布流量切换校验器

**题目要求：**
模拟蓝绿发布：给定 `blue_healthy`、`green_healthy` 与 `active_slot`，编写 `switch_traffic(active_slot, blue_healthy, green_healthy)`，仅当目标槽位健康时才允许切换，否则拒绝切换并保持原流量。

**思路提示：** 目标槽位为 `'green'` if `active_slot == 'blue'` else `'blue'`，判断目标槽位健康状态。


In [ ]:
def switch_traffic(active_slot: str, blue_healthy: bool, green_healthy: bool) -> dict:
    target = "green" if active_slot == "blue" else "blue"
    target_ok = green_healthy if target == "green" else blue_healthy
    if not target_ok:
        return {"switched": False, "active": active_slot, "reason": f"目标槽位 {target} 未通过健康检查，拒绝切流"}
    return {"switched": True, "active": target, "reason": f"已平滑将 100% 流量从 {active_slot} 切至 {target}"}


print(switch_traffic("blue", True, True))
print(switch_traffic("blue", True, False))


**答案解析与评分标准：**

蓝绿发布的核心保障是'新版本未就绪绝不切流，切流后异常秒级切回老槽位'。


---

## 🎯 老陈收尾复盘

漂亮！今天生成的 Dockerfile 是非 root 多阶段瘦身版，Nginx 关掉了流式缓冲并拉长了读超时，运维巡检器也把端口、依赖健康、内存与错误率全串起来了。明天咱们迎来第五阶段周测 + 企业安全合规与 Token 熔断！
